## Create an ensemble voting classifier by manually implementing hard and soft voting mechanisms, dynamically weighting model contributions based on performance metrics without using predefined ensemble libraries.

In [1]:
import warnings; warnings.filterwarnings('ignore')  
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC

df = pd.read_csv('heart.csv')                         
X, y = df.drop(columns='target').values.astype(float), df['target'].values
print(X.shape, '| positive share:', y.mean().round(2))

(303, 13) | positive share: 0.54


In [3]:
def auc(t, s):
    sp, sn = s[t == 1], s[t == 0]
    return (sp[:, None] > sn[None, :]).mean() + 0.5 * (sp[:, None] == sn[None, :]).mean()

def logloss(t, p):
    p = np.clip(p, 1e-6, 1 - 1e-6); return -np.mean(t * np.log(p) + (1 - t) * np.log(1 - p))

def metrics(t, p):                                    
    return dict(acc=np.mean((p > 0.5) == t), auc=auc(t, p), logloss=logloss(t, p))

In [4]:
models = {
    'LR':   lambda: LogisticRegression(max_iter=1000),
    'kNN':  lambda: KNeighborsClassifier(15),
    'NB':   lambda: GaussianNB(),
    'Tree': lambda: DecisionTreeClassifier(max_depth=4, min_samples_leaf=5, random_state=0),
    'SVM':  lambda: SVC(probability=True, random_state=0),
}
names = list(models)
def fit_predict(name, A, t, B): return models[name]().fit(A, t).predict_proba(B)[:, 1]

In [5]:
def hard_vote(P, W):
    votes = (P > 0.5).astype(float)                  
    share = (votes * W).sum(1) / W.sum(1)              
    tie = share == 0.5
    return np.where(tie, (P * W).sum(1) / W.sum(1), share)   

def soft_vote(P, W):
    return (P * W).sum(1) / W.sum(1)                 

In [6]:
def make_folds(n, k, seed): return np.array_split(np.random.default_rng(seed).permutation(n), k)

def oof_probs(A, t, seed, k=5):
    P = np.zeros((len(t), len(names)))
    for f in make_folds(len(t), k, seed):
        tr = np.setdiff1d(np.arange(len(t)), f)
        for j, n in enumerate(names): P[f, j] = fit_predict(n, A[tr], t[tr], A[f])
    return P                                         

def global_weights(P, t):
    acc = ((P > 0.5) == t[:, None]).mean(0)
    return np.maximum(acc - 0.5, 1e-3), acc         
def dynamic_weights(A, B, P, t, k=20):
    correct = ((P > 0.5) == t[:, None]).astype(float) 
    d = ((B[:, None, :] - A[None, :, :]) ** 2).sum(2) 
    nn = np.argsort(d, 1)[:, :k]                       
    return np.maximum(correct[nn].mean(1) - 0.5, 1e-3) 

In [7]:
results, gw, local_better = {}, [], []
add = lambda key, m: results.setdefault(key, []).append(m)

for rep in range(2):
    folds = make_folds(len(X), 5, rep)
    for k in range(5):
        te = folds[k]; tr = np.concatenate([folds[i] for i in range(5) if i != k])
        mu, sd = X[tr].mean(0), X[tr].std(0); A, B = (X[tr] - mu) / sd, (X[te] - mu) / sd    
        t, u = y[tr], y[te]
        P_oof = oof_probs(A, t, rep * 5 + k)
        P_te = np.c_[tuple(fit_predict(n, A, t, B) for n in names)]                          
        w, _ = global_weights(P_oof, t); gw.append(w / w.sum())
        W1, Wg, Wd = np.ones_like(P_te), np.tile(w, (len(u), 1)), dynamic_weights(A, B, P_oof, t)
        for j, n in enumerate(names): add(n, metrics(u, P_te[:, j]))
        add('Hard (equal)', metrics(u, hard_vote(P_te, W1)))
        add('Soft (equal)', metrics(u, soft_vote(P_te, W1)))
        add('Hard (global weights)', metrics(u, hard_vote(P_te, Wg)))
        add('Soft (global weights)', metrics(u, soft_vote(P_te, Wg)))
        add('Hard (dynamic weights)', metrics(u, hard_vote(P_te, Wd)))
        add('Soft (dynamic weights)', metrics(u, soft_vote(P_te, Wd)))
print('Outer folds:', len(results['Hard (equal)']))

Outer folds: 10


In [8]:
single = {n: np.mean([m['acc'] for m in results[n]]) for n in names}
best_name = max(single, key=single.get); base = np.array([m['acc'] for m in results[best_name]])
rows = []
for key, ms in results.items():
    g = lambda c: np.array([m[c] for m in ms]); d = g('acc') - base
    rows.append((key, f"{g('acc').mean():.3f} +- {g('acc').std():.3f}", f"{g('auc').mean():.3f}", '-' if key.startswith('Hard') else f"{g('logloss').mean():.3f}",
                 f"{d.mean():+.3f} +- {d.std(ddof=1) / np.sqrt(len(d)):.3f}"))
print('Best single model:', best_name)
pd.DataFrame(rows, columns=['Model', 'Accuracy', 'AUC', 'Log loss', f'Acc diff vs {best_name}'])

Best single model: kNN


,Model,Accuracy,AUC,Log loss,Acc diff vs kNN
0,LR,0.817 +- 0.051,0.906,0.407,-0.016 +- 0.013
1,kNN,0.833 +- 0.076,0.910,0.425,+0.000 +- 0.000
2,NB,0.823 +- 0.050,0.897,0.567,-0.010 +- 0.014
3,Tree,0.766 +- 0.067,0.840,1.061,-0.068 +- 0.017
4,SVM,0.822 +- 0.038,0.902,0.417,-0.011 +- 0.016
5,Hard (equal),0.827 +- 0.057,0.883,-,-0.007 +- 0.009
6,Soft (equal),0.833 +- 0.062,0.916,0.387,+0.000 +- 0.006
7,Hard (global weights),0.827 +- 0.057,0.885,-,-0.007 +- 0.009
8,Soft (global weights),0.832 +- 0.066,0.916,0.385,-0.002 +- 0.005
9,Hard (dynamic weights),0.827 +- 0.065,0.885,-,-0.007 +- 0.008


In [9]:
print('Global weights (average over folds, normalized):')
print(pd.Series(np.mean(gw, 0), index=names).round(3).to_string())
print('\nExample fold: pehle 5 test patients ke dynamic weights (normalized)')
Wn = Wd / Wd.sum(1, keepdims=True)
print(pd.DataFrame(Wn[:5], columns=names).round(2).to_string())
h, s = hard_vote(P_te, W1) > 0.5, soft_vote(P_te, W1) > 0.5
print(f'\nIs fold me hard vs soft (equal weights) {np.sum(h != s)} / {len(u)} patients pe alag nikle')

Global weights (average over folds, normalized):
LR      0.211
kNN     0.206
NB      0.208
Tree    0.168
SVM     0.207

Example fold: pehle 5 test patients ke dynamic weights (normalized)
     LR   kNN    NB  Tree   SVM
0  0.21  0.21  0.21  0.15  0.21
1  0.20  0.20  0.23  0.17  0.20
2  0.22  0.17  0.28  0.17  0.17
3  0.20  0.15  0.25  0.20  0.20
4  0.20  0.22  0.20  0.12  0.25

Is fold me hard vs soft (equal weights) 1 / 60 patients pe alag nikle
